In [ ]:
import os
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
import torch
import torch.nn as nn
import numpy as np
from model import CVmodel
from dataset import CVdataset
from loss import InfoNCELoss
import helperFunctions as HF

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
"The device is: {}".format(device)

In [ ]:
# Setup hyperparameters
NUM_EPOCHS = 12
BATCH_SIZE = 10
VEH_HIDDEN_UNITS = 256
UAV_HIDDEN_UNITS = 256
LEARNING_RATE = 1e-4
rowGrid=(0, 1023, 0)
colGrid=(0, 1023, 0)
current_dir = os.getcwd() 
root = current_dir + '\\dataset\\'
saving_dir = 'C:\\saved models\\'
scenarioList = ['FiveWays', 'FourWays', 'Park', 'Roundabout', 'StraightRoad', 'TJunction']
numRealizations = 3
Height=256
Width=1280
Length=512
train = True
trainSize = 0.9
loadSavedModel=False
lambdaReg=0
alpha=None

In [ ]:
vehImageTransform_test = transforms.Compose([
    transforms.Resize([Height, Width]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
])

uavImageTransform_test = transforms.Compose([
    transforms.Resize([Length, Length]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
])
vehImageTransform_train = transforms.Compose([
    transforms.Resize([Height, Width]),
    transforms.RandomChoice([transforms.ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2),hue=(-0.1, 0.1)),
                             transforms.RandomGrayscale(p=0.1),
                             transforms.GaussianBlur(kernel_size=(5,5), sigma=(0.1, 2.0)),
                             transforms.RandomInvert(),
                             transforms.RandomEqualize(p=0.1)],[1,1,1,1,1]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
])

uavImageTransform_train = transforms.Compose([
    transforms.Resize([Length, Length]),
    transforms.RandomChoice([transforms.ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2),hue=(-0.1, 0.1)),
                             transforms.RandomGrayscale(p=0.1),
                             transforms.GaussianBlur(kernel_size=(5,5), sigma=(0.1, 2.0)),
                             transforms.RandomInvert(),
                             transforms.RandomEqualize(p=0.1)],[1,1,1,1,1]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
])

In [ ]:
if train is True:
    cvDataset_train = CVdataset(root=root,
                            scenarioList=scenarioList,
                            numRealizations=numRealizations,
                            transform=(vehImageTransform_train, uavImageTransform_train),
                            rowGrid=rowGrid,
                            colGrid=colGrid,
                            train=train,
                            trainSize=trainSize)
    cvDataset_test = CVdataset(root=root,
                           scenarioList=scenarioList,
                           numRealizations=numRealizations,
                           transform=(vehImageTransform_test, uavImageTransform_test),
                           train=not(train),
                           trainSize=trainSize)
else:
    cvDataset_test = CVdataset(root=root,
                               scenarioList=scenarioList,
                               numRealizations=numRealizations,
                               transform=(vehImageTransform_test, uavImageTransform_test),
                               train=train,
                               trainSize=0)

In [ ]:
if train is True:
    train_dataloader = DataLoader(cvDataset_train, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(cvDataset_test, batch_size=BATCH_SIZE, shuffle=False)
else:
    test_dataloader = DataLoader(cvDataset_test, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
if loadSavedModel:
    checkpoint = torch.load('checkpoint8.pth',map_location=device,weights_only=True)

In [ ]:
torch.cuda.empty_cache()
cvModel = CVmodel(device,
                  Height=Height,
                  Width=Width,
                  Length=Length,
                  veh_hiddenUnits=VEH_HIDDEN_UNITS,
                  uav_hiddenUnits=UAV_HIDDEN_UNITS)
if loadSavedModel:
    cvModel.load_state_dict(checkpoint['model_state_dict'], strict=False)
cvModel.to(device)
for param in cvModel.parameters():
    param.requires_grad = True

In [ ]:
params = [p for p in cvModel.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=LEARNING_RATE, betas=(0.9, 0.999))
if loadSavedModel:
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
loss_fn = InfoNCELoss(alpha=None)

In [ ]:
results = HF.train(model=cvModel,
                   train_dataloader=train_dataloader, 
                   test_dataloader=val_dataloader, 
                   optimizer=optimizer,
                   loss_fn=loss_fn,
                   epochs=NUM_EPOCHS,
                   Length=int(Length/32), # Extracted feature map size by EfficientNet is Input size divided by 32.
                   batchSize=BATCH_SIZE,
                   device=device,
                   saving_dir=saving_dir,
                   lambdaReg=lambdaReg)
results